In [ ]:
import os
import sys
import jax
jax.config.update("jax_enable_x64", True)

import numpy as np

sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))

from src.config import ExperimentConfig
from src.physics.operators import build_two_qubit_base, assemble_H0, build_true_H0
from src.analysis.metrics import cz_process_fidelity, cz_comp_block, bell_from_cz_block
from src.devices.qutip_device import HiddenDevice
from src.experiments.process_tomography import run_process_tomography

config = ExperimentConfig.load("configs/two_qubit_CZ_zz.yaml")
dt = config.pulse.dt_ns
hw = config.hardware

# Twin: fitted parameters from notebook 01
H_anharm, H_int_base, Hz_control = build_two_qubit_base(levels=hw.levels)
cal_g, cal_d, cal_a = np.load(config.artifact_path("cal_params.npy"))
H0_cal = assemble_H0(H_anharm, H_int_base, Hz_control, cal_g, cal_d, cal_a)

pulse = np.load(config.artifact_path("cal_pulse_rad.npy"))

In [ ]:
H0_true, _ = build_true_H0(config)   # exact device Hamiltonian incl. unmodeled ZZ, no decoherence

rows = {"twin": H0_cal, "oracle device": H0_true}
for name, H0 in rows.items():
    U = cz_comp_block(pulse, dt, H0, Hz_control)
    r = cz_process_fidelity(U)
    print(f"{name:14s} F_avg={r['F_avg']*100:.3f}%  F_pro={r['F_pro']*100:.3f}%  "
          f"leakage={r['leakage']*100:.3f}%  Bell (ideal 1q gates)={bell_from_cz_block(U)*100:.3f}%")

In [ ]:
# Validation Test: Noiseless device (T1=0, T2=0, n_shots=None) vs. Oracle
dev_noiseless = HiddenDevice.from_config(config, t1_ns=0.0, t2_ns=0.0)
choi_exact, _ = run_process_tomography(dev_noiseless, pulse, n_shots=None)
r_qpt_exact = cz_process_fidelity(choi_exact)

# Re-compute oracle for direct comparison
U_oracle = cz_comp_block(pulse, dt, H0_true, Hz_control)
r_oracle = cz_process_fidelity(U_oracle)

print("=== 1. QPT Method Validation (Noiseless vs. Oracle) ===")
print(f"Oracle (from H0_true) : F_avg={r_oracle['F_avg']*100:.4f}%  F_pro={r_oracle['F_pro']*100:.4f}%  leakage={r_oracle['leakage']*100:.4f}%")
print(f"QPT (noiseless data)  : F_avg={r_qpt_exact['F_avg']*100:.4f}%  F_pro={r_qpt_exact['F_pro']*100:.4f}%  leakage={r_qpt_exact['leakage']*100:.4f}%")
print(f"Max absolute diff     : dF_pro={abs(r_oracle['F_pro'] - r_qpt_exact['F_pro']):.2e}, dLeak={abs(r_oracle['leakage'] - r_qpt_exact['leakage']):.2e}\n")

# Open-System Device (with T1, T2 decoherence, infinite shots)
dev_noisy = HiddenDevice.from_config(config)
choi_noisy, _ = run_process_tomography(dev_noisy, pulse, n_shots=None)
r_qpt_noisy = cz_process_fidelity(choi_noisy)

print("=== 2. Open-System Device Process Tomography (With T1/T2 Decoherence) ===")
print(f"QPT (T1/T2, inf shots): F_avg={r_qpt_noisy['F_avg']*100:.3f}%  F_pro={r_qpt_noisy['F_pro']*100:.3f}%  leakage={r_qpt_noisy['leakage']*100:.3f}%")
print(f"Decoherence penalty   : dF_avg={(r_qpt_exact['F_avg'] - r_qpt_noisy['F_avg'])*100:.3f}%  dF_pro={(r_qpt_exact['F_pro'] - r_qpt_noisy['F_pro'])*100:.3f}%\n")

# Shot-Noise Convergence Check
print("=== 3. Shot-Noise Convergence (10 independent trials per n_shots) ===")
for n_shots in [500, 2000, 10000]:
    f_pros = []
    for trial in range(10):
        choi_shot, _ = run_process_tomography(dev_noisy, pulse, n_shots=n_shots, seed=100 + trial)
        f_pros.append(cz_process_fidelity(choi_shot)["F_pro"] * 100)
    print(f"n_shots={n_shots:5d} -> F_pro = {np.mean(f_pros):.3f}% ± {np.std(f_pros):.3f}%")

In [ ]:
import matplotlib.pyplot as plt
from src.control.waveforms import build_local_drive_operators
from src.experiments.rb import run_rb_experiment
from src.calibration.rb_calibration import calibrate_rb_primitives

# Calibrate the elementary DRAG and CZ Virtual-Z primitives on the Digital Twin
Hx1, Hy1, Hx2, Hy2 = build_local_drive_operators(levels=hw.levels)
primitives = calibrate_rb_primitives(
    config, H0_cal, Hx1, Hy1, Hx2, Hy2, Hz_control,
    cal_a, cal_d, pulse, drag_duration_ns=20.0
)

# Run Reference 1Q RB and Interleaved CZ RB on the noisy HiddenDevice
lengths = [2, 4, 8, 16, 32, 64, 96, 128, 192, 256, 384]
rb_1q = run_rb_experiment(dev_noisy, primitives, lengths=lengths, n_seeds=8, mode="1q", seed=42)
rb_cz = run_rb_experiment(dev_noisy, primitives, lengths=lengths, n_seeds=8, mode="cz", seed=42)

# Isolate the Interleaved CZ Gate Fidelity: p_CZ = p_IRB / p_1Q
p_cz_isolated = min(rb_cz["p"] / rb_1q["p"], 1.0)
F_avg_cz_rb = (3.0 * p_cz_isolated + 1.0) / 4.0

print("=== Step 3: Randomized Benchmarking on Noisy HiddenDevice ===")
print(f"1Q Reference RB   : p_1Q  = {rb_1q['p']:.5f} ± {rb_1q['p_std']:.5f}  -> F_avg(1Q layer) = {rb_1q['F_avg']*100:.3f}%")
print(f"Interleaved CZ RB : p_IRB = {rb_cz['p']:.5f} ± {rb_cz['p_std']:.5f}  -> F_avg(1Q + CZ)  = {rb_cz['F_avg']*100:.3f}%")
print(f"Isolated CZ Gate  : p_CZ  = {p_cz_isolated:.5f}            -> F_avg(CZ, RB)   = {F_avg_cz_rb*100:.3f}%")
print(f"Comparison        : QPT F_avg = {r_qpt_noisy['F_avg']*100:.3f}%  |  Oracle F_avg = {r_oracle['F_avg']*100:.3f}%")
print(f"Fitted RB decay curves: 1Q RB vs. Interleaved CZ RB: A = {rb_1q['A']:.3f} vs. {rb_cz['A']:.3f}, B = {rb_1q['B']:.3f} vs. {rb_cz['B']:.3f}\n")

In [ ]:
# Plot the RB Survival Probability Decay Curves
fig, ax = plt.subplots(figsize=(7, 4.5))
m_fine = np.linspace(0, max(lengths), 200)

for label, res, color in [
    ("1Q Reference RB", rb_1q, "tab:blue"),
    ("Interleaved (1Q + CZ) RB", rb_cz, "tab:orange"),
]:
    ax.errorbar(
        res["lengths"], res["mean_surv"], yerr=res["std_surv"] / np.sqrt(len( res["surv_matrix"][0])),
        fmt=".", color=color, capsize=2, label=f"{label} (F={res['F_avg']*100:.2f}%)", linewidth=1
    )
    ax.plot(m_fine, res["A"] * (res["p"] ** m_fine) + res["B"], "--", color=color, linewidth=1)

ax.set_xlabel("Sequence Length $m$ (Clifford Layers)")
ax.set_ylabel("Survival Probability $P(|00\\rangle)$")
ax.set_title("Randomized Benchmarking Decay on Noisy HiddenDevice")
ax.grid(True, alpha=0.3)
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
for m, leak in zip(rb_cz["lengths"], rb_cz["mean_leak"]):
    print(f"m={m:3d} -> computational leakage = {leak*100:.2f}%")